# Clase 6 — Geoprocesamiento: construir variables territoriales

**Sistemas de Información Geográfica**
Especialización en Ciencias Sociales Computacionales — Universidad Nacional Guillermo Brown

| | |
|---|---|
| **Unidad del programa** | 6 — Análisis espacial |
| **Duración** | 3 horas |
| **Versión** | 2026.1 |
| **Docente** | Renzo Polo |
| **Licencia** | CC BY-SA 4.0 |

---

## 1. La pregunta de hoy

> ### ¿Qué tiene cada barrio alrededor, y cuánto de él queda lejos de un centro de salud?

Hasta acá aprendimos a conseguir datos (Clase 3), a ponerlos en el sistema de coordenadas
correcto (Clase 4) y a dibujarlos sin engañar (Clase 5). Pero en los tres casos el dato ya
venía hecho: alguien había contado los hogares con NBI y nosotros lo mapeábamos.

Hoy cambia el rol. Hoy **el dato lo fabricamos nosotros**.

Vamos a trabajar sobre los dos barrios de la Ciudad de Buenos Aires que descargamos de
OpenStreetMap en la Clase 3 —**Recoleta** y **Villa Lugano**— y vamos a construir, para cada
uno, variables que no existen en ninguna tabla: cuántas farmacias tiene por kilómetro
cuadrado, qué porcentaje de su superficie está a menos de 500 metros de un efector público
de salud, qué proporción de su población tiene 65 años o más.

Esas variables se llaman **territoriales** porque no se miden: se **calculan** a partir de la
posición relativa de las cosas. Y son las que después entran en un modelo, en una tabla de un
informe o en el mapa de una tesis.

## 2. Objetivos de esta clase

Al terminar, deberías poder:

1. **Unir** una tabla sin geometría a una capa geográfica por una clave común, y verificar
   que la unión salió bien.
2. **Geocodificar** un listado de direcciones, y **medir** el error que esa operación
   introduce.
3. **Contar y agregar** elementos de una capa dentro de las unidades de otra con una unión
   espacial (`sjoin`).
4. **Construir** un área de influencia (`buffer`), disolverla y calcular qué porcentaje de una
   unidad queda cubierto.
5. **Armar** una tabla de variables territoriales con una fila por unidad de análisis, y
   exportarla.
6. **Justificar** en qué sistema de coordenadas se hace cada medición, y qué supone cada
   variable construida.

**Qué NO cubre esta clase.** Distancias y accesibilidad —a qué distancia está el
establecimiento más cercano, cuánto se tarda en llegar caminando— son el tema de la Clase 7.
Hoy medimos *adentro* y *alrededor*; la semana que viene medimos *hasta*.

## 3. Material de esta clase

Esta clase no tiene presentación: la teoría está acá, en la sección 4, y después va todo a la
práctica.

| Bloque | Qué retoma de clases anteriores |
|---|---|
| Unir por clave | Clase 3 — el CSV de poblaciones.org, que bajamos del portal |
| Geocodificación | Clase 3 — los sesgos de cobertura de OpenStreetMap |
| Unión espacial y conteos | Clase 2 — el conteo de escuelas por celda y el problema de la unidad de análisis (MAUP) |
| Áreas de influencia | Clase 4 — por qué medir en metros exige un CRS proyectado |
| Densidades y normalización | Clase 5 — conteo, porcentaje y densidad responden preguntas distintas |

**Datos.** Todos salen del repositorio del curso. Los cuatro que usamos hoy:

| Archivo | Contenido | Fuente |
|---|---|---|
| `osm_barrios_limites.gpkg` | Contorno de Recoleta y Villa Lugano | OpenStreetMap (Clase 3) |
| `osm_amenities_barrios.gpkg` | 1.770 equipamientos con etiqueta `amenity` | OpenStreetMap (Clase 3) |
| `ign_salud.gpkg` | 8.311 establecimientos de salud del país | IGN, geoservicio WFS (Clase 3) |
| `indicadores_hogares_departamentos_2022.gpkg` | 527 departamentos con indicadores de hogares | INDEC, Censo 2022 (Clase 5) |
| `poblaciones_departamentos_2022.csv` | El CSV crudo del portal, con 44 columnas | poblaciones.org (Clase 3) |

---

## 4. Qué es una variable territorial

Una **variable territorial** es un atributo de una unidad del espacio —un barrio, un radio
censal, un departamento— que **no viene en ningún registro administrativo** y que se obtiene
poniendo dos capas una encima de la otra.

"Población del barrio" no es una variable territorial: la contó el censo. "Cantidad de
farmacias del barrio" sí lo es: nadie la contó, pero tenemos la capa de barrios y la capa de
farmacias, y de la relación entre las dos sale el número.

### 4.1 Las tres preguntas del geoprocesamiento

Casi todo lo que se hace con dos capas responde a una de estas tres preguntas.

| Pregunta | Operación | Ejemplo | Dónde se ve |
|---|---|---|---|
| **¿Qué hay adentro?** | Unión espacial (`sjoin`) y agregación | Cuántas escuelas tiene cada barrio | Hoy, bloque 7 |
| **¿Qué hay alrededor?** | Área de influencia (`buffer`) y superposición (`overlay`) | Qué parte del barrio está a menos de 500 m de un hospital | Hoy, bloque 8 |
| **¿A qué distancia está?** | Distancia y vecino más cercano (`sjoin_nearest`) | Cuánto hay hasta el centro de salud más próximo | Clase 7 |

Y antes de cualquiera de las tres hay una operación previa, que no es espacial pero sin la
cual no se empieza: **unir por clave** la tabla que trae los datos con la capa que trae la
geometría. Es la que más se usa y la que más silenciosamente falla. Empezamos por ahí.

### 4.2 Adónde queremos llegar: la tabla de análisis

Todo el trabajo de hoy apunta a una tabla con esta forma:

| barrio | superficie_km2 | farmacias_por_km2 | escuelas_por_km2 | cobertura_salud_500m_perc |
|---|---|---|---|---|
| Recoleta | … | … | … | … |
| Villa Lugano | … | … | … | … |

**Una fila por unidad de análisis y una columna por variable.** Esa es la tabla que después
entra en un modelo, en un gráfico o en un mapa. Todo lo que hagamos hoy es agregarle
columnas.

Dos reglas de la casa, que vamos a sostener toda la clase:

- **La unidad va en el nombre.** `superficie_km2`, no `superficie`. `distancia_m`, no
  `distancia`. Dentro de seis meses no vas a acordarte.
- **Lo relativo, no lo absoluto.** Recoleta tiene 6,9 km² y Villa Lugano 9,3 km². Comparar
  conteos crudos entre unidades de distinto tamaño es el error de la Clase 5, ahora del lado
  de la producción del dato y no del dibujo.

### 4.3 Por qué esto exige un CRS proyectado

Esta es la clase donde la Clase 4 se cobra la deuda.

Todas las operaciones de hoy —superficies, áreas de influencia de 500 metros, densidades por
kilómetro cuadrado— son **mediciones**. Y en coordenadas geográficas (EPSG:4326) la unidad es
el **grado**, que no es una unidad de longitud: un grado de longitud mide 111 km en el
Ecuador y 0 en el polo.

Un `buffer(500)` sobre datos en EPSG:4326 no da 500 metros. Da 500 **grados**, algo así como
cinco vueltas al mundo, y GeoPandas lo calcula sin quejarse más allá de una advertencia.

Para la Ciudad de Buenos Aires el sistema correcto es **EPSG:5347**, POSGAR 2007 faja 5, cuyo
meridiano central es −58,5° —la ciudad está en −58,4°—. Sus unidades son metros.

> ⚠️ **Regla de la clase:** antes de toda operación métrica, comprobar el CRS. Vamos a
> escribir esa comprobación como código, no como comentario.

---

## 5. Preparación

### ▶️ Las bibliotecas

`geopy` es nueva: es la que le habla a los servicios de geocodificación.

In [ ]:
!pip install -q "geopandas==1.0.1" "mapclassify==2.8.1" "matplotlib==3.9.2" \
               "folium==0.17.0" "geopy==2.4.1"

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

print("Bibliotecas listas")

### ▶️ Los datos y los sistemas de coordenadas

Definimos de entrada los dos sistemas que vamos a usar y por qué cada uno:

- **EPSG:4326** — coordenadas geográficas. Es como vienen los datos y es lo que entienden los
  mapas web. Se usa para **mostrar**.
- **EPSG:5347** — POSGAR 2007 faja 5, en metros. Se usa para **medir**.

In [ ]:
DATOS = "https://raw.githubusercontent.com/renzoepolo/sig-ciencias-sociales/main/datos/"

GEOGRAFICO = "EPSG:4326"   # para mostrar
METRICO    = "EPSG:5347"   # para medir: POSGAR 2007 faja 5, en metros

barrios = gpd.read_file(DATOS + "osm_barrios_limites.gpkg")
print(barrios.crs)
barrios

### ▶️ Una función de comprobación

La vamos a llamar antes de cada medición. Son cuatro líneas y evitan el error más caro de
todo el curso.

In [ ]:
def comprobar_metrico(capa, nombre="la capa"):
    """Verifica que la capa esté en un CRS proyectado con unidades en metros."""
    assert capa.crs is not None, f"{nombre} no tiene CRS declarado"
    assert capa.crs.is_projected, f"{nombre} está en {capa.crs.name}: no se puede medir"
    unidad = capa.crs.axis_info[0].unit_name
    assert unidad == "metre", f"{nombre} mide en {unidad}, no en metros"
    print(f"✓ {nombre}: {capa.crs.name} — medición en metros")


comprobar_metrico(barrios.to_crs(METRICO), "barrios")

---

## 6. Unir una tabla a una geometría

### 🧭 Concepto

La situación es la de siempre: tenés una capa con las geometrías —los departamentos, los
radios censales, los barrios— y, aparte, una planilla con los datos que te interesan. Ninguna
de las dos sirve sola.

La operación que las junta es un **join por clave** (`merge` en pandas): se elige una columna
que esté en las dos tablas y que identifique unívocamente a cada unidad, y pandas empareja
fila con fila.

En la Argentina esa clave suele ser el **código del INDEC**: cinco dígitos, dos de provincia
y tres de departamento. La Comuna 1 de la Ciudad es `02007`.

Y ahí está el problema de esta sección.

### ▶️ Las dos piezas

Del lado de la geometría, los 527 departamentos de la Clase 5. Del lado de la tabla, el CSV
de poblaciones.org **tal como se baja del portal**, con sus 44 columnas, que ya vimos en la
Clase 3.

In [ ]:
deptos = gpd.read_file(DATOS + "indicadores_hogares_departamentos_2022.gpkg")
censo  = pd.read_csv(DATOS + "poblaciones_departamentos_2022.csv")

print("Departamentos (geometría):", len(deptos), "filas,", len(deptos.columns), "columnas")
print("Censo (tabla):            ", len(censo),  "filas,", len(censo.columns),  "columnas")

El CSV trae la población por grupo de edad, que el GeoPackage no tiene. Esa es la columna que
queremos traernos: **cuánta gente de 65 años o más vive en cada departamento**.

La primera columna trae el código del departamento y debería llamarse `dpto`. Fijate cómo se
llama en realidad.

In [ ]:
print(repr(censo.columns[0]))

`'﻿dpto'`. Hay un carácter invisible pegado adelante: es un **BOM** (*byte order mark*),
que algunos programas —Excel, sobre todo— escriben al principio de los CSV en UTF-8 para
declarar la codificación. pandas lo lee como parte del nombre, así que `censo["dpto"]` daría
`KeyError`.

Normalmente esto se arregla al leer, con `encoding="utf-8-sig"`, que descarta el BOM. **Acá
no alcanza**, y vale la pena ver por qué: si mirás el archivo crudo, el encabezado empieza con
`"﻿dpto","Sexo",…`. El BOM quedó **adentro de las comillas**, después del primer carácter, y
`utf-8-sig` sólo descarta el que está al principio del archivo.

Cuando la codificación no salva, se renombra por posición. No es elegante, pero es explícito.

In [ ]:
censo = censo.rename(columns={censo.columns[0]: "dpto"})

censo[["dpto", "Nombre de departamentos/comuna", "Población total",
       "Poblacion 65 años o más"]].head()

### ▶️ El join que no funciona

Las dos tablas tienen el código. Deberíamos poder unirlas directamente.

In [ ]:
# ⚠️ Esta celda da error a propósito. Leé el mensaje antes de seguir.
deptos.merge(censo[["dpto", "Poblacion 65 años o más"]],
             left_on="codigo", right_on="dpto", how="left")

### 🔍 Qué pasó

> `ValueError: You are trying to merge on object and int64 columns`

pandas se niega, y hace bien. Miremos las dos claves:

In [ ]:
print("codigo (GeoPackage):", deptos["codigo"].dtype, "→", deptos["codigo"].head(3).tolist())
print("dpto   (CSV):       ", censo["dpto"].dtype,    "→", censo["dpto"].head(3).tolist())

El GeoPackage guardó el código como **texto**: `'02007'`. El CSV lo trajo como **número**:
`2007`. Al leer el archivo, pandas vio una columna de dígitos, decidió que era un entero y
**se comió el cero de la izquierda**.

Esto es el error más frecuente de todo el trabajo con datos territoriales argentinos, y tiene
dos versiones:

- La **ruidosa**, que es esta: los tipos no coinciden y pandas frena.
- La **silenciosa**, que es peor: las dos columnas son enteros, el join corre sin protestar y
  une mal —o no une nada— sin que nadie se entere. Pasa cuando la geometría también perdió el
  cero.

La regla, entonces: **un código de identificación no es un número, es una etiqueta.** No se
suma, no se promedia, y se lee como texto siempre.

### ▶️ La corrección

Hay dos caminos, y conviene conocer los dos.

**El bueno: declarar el tipo al leer.** `pd.read_csv(..., dtype={"codigo": str})` obliga a
pandas a leer esa columna como texto y el cero nunca se pierde. Es lo que hay que hacer
siempre que se pueda, y no se puede acá: para nombrar la columna en `dtype` habría que saber
su nombre, y su nombre tiene el BOM adentro.

**El de rescate: recomponer el código.** Los códigos del INDEC tienen cinco dígitos, así que
rellenamos con ceros a la izquierda hasta llegar a cinco.

In [ ]:
censo["dpto"] = censo["dpto"].astype(str).str.zfill(5)

print("Ahora:", censo["dpto"].dtype, "→", censo["dpto"].head(3).tolist())

> ⚠️ `zfill` es un parche y tiene su límite: funciona porque **todos** los códigos de esta
> tabla tienen el mismo largo. Si la clave fuera de largo variable —un CUIT, un código postal
> mezclado con letras— rellenar a ciegas inventaría datos. Antes de usarlo, comprobá el largo:
> `censo["dpto"].str.len().value_counts()`.

In [ ]:
deptos = deptos.merge(
    censo[["dpto", "Poblacion 65 años o más"]],
    left_on="codigo", right_on="dpto",
    how="left",
    validate="1:1",     # exige que cada fila de un lado tenga a lo sumo una del otro
)

deptos = deptos.rename(columns={"Poblacion 65 años o más": "poblacion_65_mas"})
deptos[["codigo", "departamento", "poblacion", "poblacion_65_mas"]].head()

### ✅ Comprobación

Un `merge` con `how="left"` **nunca falla**: los que no encuentran pareja quedan en blanco.
Por eso hay que contarlos, siempre, antes de seguir.

In [ ]:
sin_pareja = deptos["poblacion_65_mas"].isna().sum()

print(f"Filas: {len(deptos)} (esperadas: 527)")
print(f"Sin coincidencia en el censo: {sin_pareja}")
assert sin_pareja == 0, "Hay departamentos sin dato: revisar la clave antes de seguir"

### ▶️ Y ahora sí, la variable

Con las dos tablas unidas, la variable nueva es una división. Y es **relativa**, no un
conteo: el porcentaje de la población que tiene 65 años o más.

In [ ]:
deptos["perc_65_mas"] = deptos["poblacion_65_mas"] / deptos["poblacion"] * 100

deptos.nlargest(5, "perc_65_mas")[["departamento", "provincia", "poblacion", "perc_65_mas"]]

### 🔍 Interpretación

El departamento más envejecido del país es la **Comuna 2 de la Ciudad de Buenos Aires**, con
el **20,3 %** de su población de 65 años o más, contra un promedio nacional del 11,4 %.

La Comuna 2 **es** el barrio de Recoleta: los dos límites coinciden. Y en el otro extremo de
la ciudad, la Comuna 8 —donde está Villa Lugano— tiene el 11,1 %, casi la mitad.

Guardemos ese número, porque al final de la clase va a chocar con otro.

In [ ]:
# Las dos comunas que nos interesan hoy
deptos.loc[deptos["codigo"].isin(["02014", "02056"]),
           ["codigo", "departamento", "poblacion", "perc_65_mas"]]

---

## 7. De la dirección al punto: geocodificación

### 🧭 Concepto

**Geocodificar** es convertir una descripción textual de un lugar —una dirección, el nombre
de una institución— en un par de coordenadas.

Es la puerta de entrada de casi toda investigación social al territorio: las encuestas traen
domicilios, los registros administrativos traen direcciones, los padrones traen nombres de
establecimientos. Nada de eso es un punto todavía.

El servicio que vamos a usar es **Nominatim**, el geocodificador de OpenStreetMap: es
gratuito, no pide registro y acepta un pedido por segundo. Lo que devuelve depende
enteramente de lo que la comunidad de OSM haya cargado, y eso —lo vimos en la Clase 3— **no
está repartido de manera pareja en el territorio**.

Hoy vamos a poner número a esa afirmación.

### ▶️ Los establecimientos de salud de los dos barrios

Son 13, recortados del geoservicio del IGN: cuatro en Recoleta y nueve en Villa Lugano. Nos
importan por dos motivos: son los que vamos a geocodificar en este bloque, y son los que van
a generar las áreas de influencia del bloque 8.

Como vienen del IGN con su coordenada, tenemos **referencia**: podemos comparar contra ella y
medir cuánto se equivocó el geocodificador.

In [ ]:
salud = gpd.read_file(DATOS + "salud_barrios.gpkg")

print(salud["barrio"].value_counts().to_string())
salud[["nombre", "tipo", "barrio"]]

### ▶️ Geocodificar por nombre

Le vamos a pedir a Nominatim los 13, por su nombre completo más el barrio y la ciudad.

`RateLimiter` respeta el segundo de espera entre pedidos que exige el servicio: sin él, el
servidor nos bloquea. Son 13 consultas, así que la celda tarda unos 20 segundos.

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# El user_agent es obligatorio y tiene que identificar a quien consulta
nominatim = Nominatim(user_agent="sig-unab-2026-clase6")
geocodificar = RateLimiter(nominatim.geocode, min_delay_seconds=1.1)

salud["consulta"] = (salud["nombre"] + ", " + salud["barrio"]
                     + ", Ciudad Autónoma de Buenos Aires, Argentina")

resultados = salud["consulta"].apply(geocodificar)
salud["encontrado"] = [r.address if r else None for r in resultados]
salud["lon_geo"]    = [r.longitude if r else None for r in resultados]
salud["lat_geo"]    = [r.latitude  if r else None for r in resultados]

print(f"Resolvió {salud['lat_geo'].notna().sum()} de {len(salud)}")

### ✅ Comprobación — cuántos resolvió, y dónde

In [ ]:
salud["resuelto"] = salud["lat_geo"].notna()

salud.groupby("barrio")["resuelto"].agg(["sum", "count"])

### ▶️ Cuánto se equivocó

Para los que sí resolvió, medimos la distancia entre el punto que devolvió Nominatim y el
punto del IGN. Es una medición: va en CRS métrico.

In [ ]:
geocodificados = gpd.GeoDataFrame(
    salud,
    geometry=gpd.points_from_xy(salud["lon_geo"], salud["lat_geo"]),
    crs=GEOGRAFICO,
)

comprobar_metrico(salud.to_crs(METRICO), "referencia del IGN")

salud["error_m"] = (salud.to_crs(METRICO).geometry
                    .distance(geocodificados.to_crs(METRICO).geometry)
                    .round())

salud.loc[salud["resuelto"], ["barrio", "nombre", "error_m", "encontrado"]]

### 🔍 Interpretación — el error no está repartido parejo

El resultado de esta celda es el contenido del bloque.

Nominatim resolvió **los dos hospitales célebres de Recoleta** —el Rivadavia y el Hospital de
Clínicas— con un error de unas pocas decenas de metros, que para casi cualquier análisis es
despreciable. Y no resolvió **ninguno de los nueve CeSAC de Villa Lugano**.

No es que los CeSAC no existan ni que estén mal escritos: es que nadie los cargó en
OpenStreetMap con ese nombre. El Hospital de Clínicas tiene página propia, turismo, fotos y
una docena de colaboradores que lo editaron. El CeSAC Nº 29 atiende a un barrio que no genera
ese tipo de tráfico.

**Esta es la forma concreta que toma el sesgo de las fuentes colaborativas en un análisis
territorial.** Si hubiéramos armado la capa de efectores de salud geocodificando un padrón,
habríamos terminado con un mapa donde Recoleta tiene servicios y Villa Lugano no tiene
ninguno —y la conclusión habría sido exactamente la contraria a la verdadera, como vamos a
ver en el bloque 8—.

Tres consecuencias prácticas:

1. **La tasa de éxito de una geocodificación es un resultado**, no un detalle técnico. Se
   informa: "se geocodificaron N de M registros".
2. **Los que fallan no fallan al azar.** Antes de descartarlos hay que mirar si tienen algo en
   común, porque casi siempre lo tienen.
3. **Si existe una fuente oficial con coordenadas, se usa ésa.** Geocodificar es el último
   recurso, no el primero.

> 🤖 **Actividad de IA.** Pedile a un asistente las coordenadas del CeSAC Nº 29 de Villa
> Lugano. Va a darte un par de números con total seguridad y varios decimales. Compará contra
> la coordenada del IGN que tenemos en `salud`. ¿A cuántos metros está? Un número plausible y
> bien formateado no es un dato verificado.

---

## 8. ¿Qué hay adentro? Unión espacial y agregación

### 🧭 Concepto

La **unión espacial** (`gpd.sjoin`) es el `merge` de la geografía: en lugar de emparejar filas
por una columna en común, las empareja por **posición**.

No hace falta que las dos capas compartan ningún atributo. Alcanza con que compartan el
espacio y con declarar la relación que buscamos, el **predicado**:

| Predicado | Empareja cuando… | Uso típico |
|---|---|---|
| `within` | la geometría de la izquierda está adentro de la de la derecha | puntos en polígonos |
| `intersects` | se tocan o se superponen, aunque sea en un borde | el más laxo, y el que está por defecto |
| `contains` | la de la izquierda contiene a la de la derecha | polígonos que encierran puntos |

> ⚠️ Los dos requisitos que se olvidan: las dos capas tienen que estar **en el mismo CRS**, y
> `sjoin` **no mide nada**, sólo relaciona. Por eso éste es el único bloque de la clase que
> podría hacerse en coordenadas geográficas. Igual lo hacemos en métrico, porque enseguida
> vamos a dividir por superficie.

### ▶️ Unión 1 a 1 — ¿en qué barrio cae cada establecimiento?

Empecemos por el caso simple: cada punto cae en un polígono y en uno solo. Lo hacemos con la
capa nacional de salud, los 8.311 establecimientos del país, para quedarnos con los que caen
en nuestros dos barrios.

Es, exactamente, la operación con la que se preparó el archivo `salud_barrios.gpkg` que
usamos en el bloque anterior.

In [ ]:
salud_pais = gpd.read_file(DATOS + "ign_salud.gpkg").to_crs(METRICO)
barrios_m  = barrios.to_crs(METRICO)

comprobar_metrico(salud_pais, "salud del país")
comprobar_metrico(barrios_m, "barrios")

en_barrios = gpd.sjoin(salud_pais, barrios_m, predicate="within")

print(f"De {len(salud_pais)} establecimientos del país, {len(en_barrios)} caen en los barrios")
en_barrios["barrio"].value_counts()

Cada punto se quedó con las columnas del polígono que lo contiene —acá, `barrio`—. Eso es
todo lo que hace una unión espacial 1 a 1: **bajarle al punto el atributo del área donde
está**.

Es la operación que asigna una encuesta a su radio censal, un delito a su comuna o un
establecimiento a su partido.

### ▶️ Unión 1 a muchos — contar equipamiento por barrio

Ahora al revés: nos interesa el **barrio**, y cada barrio contiene muchos puntos. El `sjoin`
sigue siendo el mismo; lo que cambia es que después **agregamos** con `groupby`.

Usamos los 1.770 equipamientos de OpenStreetMap de la Clase 3.

In [ ]:
equip = gpd.read_file(DATOS + "osm_amenities_barrios.gpkg").to_crs(METRICO)

print(f"{len(equip)} equipamientos, {equip['amenity'].nunique()} tipos distintos")
print("Columnas:", list(equip.columns))
equip.head(3)

Ojo con la columna `barrio`: **ya viene en el archivo**, porque en la Clase 3 consultamos OSM
un barrio por vez y guardamos de cuál era cada resultado.

Si dejáramos esa columna, el `sjoin` traería otra con el mismo nombre y GeoPandas resolvería
el choque renombrando las dos —`barrio_left` y `barrio_right`—, que es la fuente de la mitad
de los errores con uniones. La sacamos y la recalculamos, que además es lo que queremos
practicar.

In [ ]:
equip = equip.drop(columns=["barrio"])

In [ ]:
equip_en_barrio = gpd.sjoin(equip, barrios_m[["barrio", "geometry"]], predicate="within")

conteo = (equip_en_barrio
          .groupby("barrio")
          .size()
          .rename("equipamientos"))

conteo

### ▶️ Agregación por tipo

`groupby` acepta más de una columna. Con eso sale la tabla de contingencia: cuántos
equipamientos de cada tipo tiene cada barrio.

In [ ]:
por_tipo = (equip_en_barrio
            .pivot_table(index="amenity", columns="barrio", aggfunc="size", fill_value=0))

por_tipo.sort_values("Recoleta", ascending=False).head(12)

### ▶️ De conteo a densidad

Los conteos no son comparables: Recoleta tiene 6,9 km² y Villa Lugano 9,3. Es el mismo
problema de la Clase 5, ahora del lado de la producción del dato.

Dividimos por la superficie, que calculamos nosotros —otra variable territorial— con el CRS
métrico ya comprobado.

In [ ]:
barrios_m["superficie_km2"] = barrios_m.area / 1_000_000

barrios_m[["barrio", "superficie_km2"]].round(2)

In [ ]:
densidad = por_tipo.T.join(barrios_m.set_index("barrio")["superficie_km2"])

for tipo in ["pharmacy", "school", "cafe", "bank", "clinic"]:
    densidad[tipo + "_por_km2"] = (densidad[tipo] / densidad["superficie_km2"]).round(1)

densidad[[c for c in densidad.columns if c.endswith("_por_km2")]]

### 🔍 Interpretación — dos distribuciones distintas

Mirá la tabla renglón por renglón, porque no dice una sola cosa.

**Farmacias:** 12,4 por km² en Recoleta contra 0,8 en Villa Lugano. **Quince veces más.**
Bancos: 8,2 contra 0,6. Cafés: 31,4 contra 0,4, casi ochenta veces.

**Escuelas: 9,2 por km² en Recoleta y 9,2 en Villa Lugano.** Idénticas.

Las farmacias, los bancos y los cafés son equipamiento **de mercado**: se instalan donde hay
capacidad de compra. Las escuelas son equipamiento **del Estado**: se instalan donde hay
población en edad escolar. Son dos lógicas de localización distintas y dejan dos huellas
territoriales distintas, y la tabla las separa sin que nadie se lo haya pedido.

> ⚠️ **Una advertencia sobre la fuente.** Acabamos de ver que OSM tiene menos cargado en
> Villa Lugano. ¿Cuánto de la diferencia en farmacias es desigualdad real y cuánto es
> subregistro? La respuesta honesta es que con estos datos no se puede separar. Que las
> escuelas den empatadas es tranquilizador —si el subregistro fuera masivo, también estarían
> subcontadas—, pero no lo resuelve. Esto se escribe en la sección de limitaciones; no se
> tapa.

---

## 9. ¿Qué hay alrededor? Áreas de influencia

### 🧭 Concepto

Un **área de influencia** o **buffer** es la zona que rodea a una geometría hasta una
distancia dada. Alrededor de un punto es un círculo; alrededor de una línea, una franja;
alrededor de un polígono, el polígono engordado.

Sirve para convertir una pregunta de proximidad en una pregunta de superficie: en lugar de
"¿está cerca?", que no se puede contestar sin definir *cerca*, preguntamos "¿qué parte del
barrio está a menos de 500 metros de un efector de salud?".

**Los 500 metros son una decisión, no un dato.** Es una caminata de unos 6 minutos, y es el
umbral que suele usarse para servicios de proximidad. Otro umbral da otro resultado, y al
final del bloque lo vamos a comprobar.

El procedimiento tiene tres pasos:

1. **Generar** un buffer alrededor de cada establecimiento.
2. **Disolver** los buffers en una sola geometría, porque se superponen y si no la superficie
   compartida se contaría dos veces.
3. **Intersecar** esa geometría con cada barrio y medir qué proporción quedó cubierta.

### ▶️ Paso 1 — El buffer

Acá es donde la comprobación de CRS deja de ser una formalidad: `buffer(500)` toma el número
en las unidades del sistema de la capa.

In [ ]:
salud_m = salud.to_crs(METRICO)
comprobar_metrico(salud_m, "establecimientos de salud")

RADIO_M = 500

areas = salud_m.copy()
areas["geometry"] = salud_m.buffer(RADIO_M)

print(f"{len(areas)} áreas de influencia de {RADIO_M} m")
print(f"Superficie de cada una: {areas.area.iloc[0] / 10_000:.1f} ha "
      f"(un círculo de 500 m tiene {3.1416 * 500**2 / 10_000:.1f} ha)")

### 👀 Paso 2 — Ver lo que hicimos

Antes de medir, mirar. Un mapa interactivo con las tres capas: el barrio, las áreas de
influencia y los establecimientos.

In [ ]:
mapa = barrios.explore(color="grey", style_kwds=dict(fill=False, weight=2),
                       tiles="CartoDB positron", name="Barrios")

areas.to_crs(GEOGRAFICO).explore(
    m=mapa, color="orange", style_kwds=dict(fillOpacity=0.3, weight=0),
    name="A menos de 500 m")

salud.explore(m=mapa, color="red", marker_kwds=dict(radius=4),
              tooltip=["nombre", "tipo"], name="Efectores de salud")

folium.LayerControl().add_to(mapa)
mapa

Se ve el problema del paso siguiente: en Villa Lugano los círculos **se superponen**, y una
parte del área aparece cubierta por dos o tres establecimientos a la vez.

Si sumáramos las superficies de los 13 círculos, esa zona contaría varias veces y el
porcentaje podría pasarse del 100 %.

### ▶️ Paso 3 — Disolver

`union_all()` funde todas las geometrías de la capa en una sola, eliminando las
superposiciones. Es la operación que evita la doble contabilidad.

In [ ]:
cobertura = areas.union_all()

suma_de_circulos = areas.area.sum() / 1_000_000
area_disuelta    = cobertura.area / 1_000_000

print(f"Suma de los 13 círculos: {suma_de_circulos:.2f} km²")
print(f"Área realmente cubierta: {area_disuelta:.2f} km²")
print(f"Se contaba de más:       {suma_de_circulos - area_disuelta:.2f} km²")

### ▶️ Paso 4 — Intersecar y medir

Ahora sí, la variable. Para cada barrio: qué parte de su superficie cae dentro de la zona
cubierta.

In [ ]:
barrios_m["area_cubierta_km2"] = (
    barrios_m.geometry.intersection(cobertura).area / 1_000_000)

barrios_m["cobertura_500m_perc"] = (
    barrios_m["area_cubierta_km2"] / barrios_m["superficie_km2"] * 100)

barrios_m[["barrio", "superficie_km2", "area_cubierta_km2",
           "cobertura_500m_perc"]].round(1)

### ✅ Comprobación

Un porcentaje de superficie tiene que estar entre 0 y 100. Si se pasa, casi siempre es porque
faltó disolver.

In [ ]:
p = barrios_m["cobertura_500m_perc"]

assert p.between(0, 100).all(), "Porcentaje fuera de rango: ¿se disolvieron los buffers?"
print(f"✓ Cobertura entre {p.min():.1f} % y {p.max():.1f} %")

### 👀 Paso 5 — Qué parte queda afuera

La pregunta del título de la clase era por lo que queda **fuera** del alcance. Es la
diferencia entre el barrio y la zona cubierta.

In [ ]:
sin_cobertura = barrios_m.copy()
sin_cobertura["geometry"] = barrios_m.geometry.difference(cobertura)

mapa = barrios.explore(color="grey", style_kwds=dict(fill=False, weight=2),
                       tiles="CartoDB positron", name="Barrios")

sin_cobertura.to_crs(GEOGRAFICO).explore(
    m=mapa, color="crimson", style_kwds=dict(fillOpacity=0.45, weight=0),
    tooltip=["barrio"], name="A más de 500 m de un efector")

salud.explore(m=mapa, color="black", marker_kwds=dict(radius=3), name="Efectores")

folium.LayerControl().add_to(mapa)
mapa

### ▶️ Paso 6 — ¿Y si el umbral fuera otro?

Los 500 metros los elegimos nosotros. Vale la pena ver cuánto dependía el resultado de esa
elección.

In [ ]:
filas = []
for radio in [300, 500, 1000, 1500]:
    zona = salud_m.buffer(radio).union_all()
    for _, b in barrios_m.iterrows():
        filas.append(dict(
            radio_m=radio,
            barrio=b["barrio"],
            cobertura_perc=round(100 * b.geometry.intersection(zona).area
                                 / b.geometry.area, 1)))

pd.DataFrame(filas).pivot(index="radio_m", columns="barrio", values="cobertura_perc")

### 🔍 Interpretación — el resultado que no esperábamos

A 500 metros, **Villa Lugano tiene el 49 % de su superficie cubierta y Recoleta el 24 %**. El
doble. A 1.000 metros la brecha se mantiene: 91 % contra 53 %.

Vale la pena detenerse, porque es lo contrario de lo que sugiere el bloque anterior.

En farmacias, bancos y cafés Recoleta le sacaba entre quince y ochenta veces de ventaja. En
cobertura de **salud pública de proximidad**, Villa Lugano gana por el doble. Los dos
resultados son correctos y hablan de cosas distintas:

- Recoleta tiene **cuatro** efectores públicos, y son **hospitales**: grandes, de alta
  complejidad, que atienden a toda la ciudad y no al barrio. Además es la comuna más
  envejecida del país —20,3 % de 65 años o más, el número del bloque 6—, o sea la población
  que más usa el sistema de salud.
- Villa Lugano tiene **nueve** efectores y son **CeSAC**: chicos, de primer nivel, pensados
  exactamente para la atención de proximidad y colocados deliberadamente donde el mercado no
  pone nada.

La política pública de atención primaria es visible en el territorio y esta variable la
detecta. Un análisis que se hubiera quedado en "cantidad de equipamiento por barrio" habría
concluido que Villa Lugano está desprovista de todo.

**Y la advertencia, que es la misma de siempre:** la cobertura es de **superficie**, no de
población. Que el 49 % del territorio de Villa Lugano esté a menos de 500 m de un CeSAC no
dice qué porcentaje de sus habitantes lo está —el barrio no tiene la gente repartida pareja—.
Tampoco dice nada sobre horarios de atención, turnos disponibles ni calidad. La variable mide
**una** dimensión del acceso: la distancia. Es mucho menos de lo que la palabra "acceso"
sugiere, y conviene decirlo en el informe antes de que lo diga otro.

---

## 10. La tabla de variables territoriales

### ▶️ Armarla

Todo el trabajo de hoy, en la forma en que sirve: una fila por unidad de análisis, una
columna por variable, la unidad de medida en el nombre.

In [ ]:
tabla = barrios_m[["barrio", "superficie_km2",
                   "area_cubierta_km2", "cobertura_500m_perc"]].copy()

tabla["efectores_salud"]   = tabla["barrio"].map(salud["barrio"].value_counts())
tabla["equipamientos"]     = tabla["barrio"].map(conteo)
tabla["farmacias_por_km2"] = tabla["barrio"].map(densidad["pharmacy_por_km2"])
tabla["escuelas_por_km2"]  = tabla["barrio"].map(densidad["school_por_km2"])

tabla.round(1)

### ▶️ Exportarla

Tres formatos, tres usos. El GeoPackage conserva la geometría y es con lo que se sigue
trabajando; el CSV es la tabla para el informe; el PNG es la figura.

In [ ]:
salida = barrios_m.to_crs(GEOGRAFICO)
salida = salida.merge(tabla.drop(columns=["superficie_km2", "area_cubierta_km2",
                                          "cobertura_500m_perc"]), on="barrio")

salida.to_file("variables_territoriales_barrios.gpkg", driver="GPKG")
tabla.round(2).to_csv("variables_territoriales_barrios.csv", index=False)

print("Guardados:")
print("  variables_territoriales_barrios.gpkg  — con geometría, para seguir trabajando")
print("  variables_territoriales_barrios.csv   — la tabla, para el informe")

> ⚠️ En Colab estos archivos quedan en el disco temporal de la sesión y **se borran al
> cerrarla**. Para conservarlos hay que bajarlos con el panel de archivos de la izquierda, o
> montar Google Drive.

---

## 11. 🧪 Tu turno

Repetí el análisis del bloque 9 cambiando **el equipamiento**: en lugar de los efectores de
salud del IGN, usá las **escuelas** de OpenStreetMap (`amenity == "school"`), que son las que
dieron idéntica densidad en los dos barrios.

La celda de abajo tiene el esqueleto. Cambiá lo que está marcado y contestá las cuatro
preguntas.

In [ ]:
# 1 · Elegí el equipamiento y el radio
TIPO   = "school"     # probá también: "pharmacy", "kindergarten", "bank"
RADIO  = 500          # en metros

elegidos = equip_en_barrio[equip_en_barrio["amenity"] == TIPO]

# 2 · Área de influencia, disuelta
zona = elegidos.buffer(RADIO).union_all()

# 3 · Porcentaje cubierto de cada barrio
resultado = barrios_m[["barrio", "superficie_km2"]].copy()
resultado["cubierto_perc"] = [
    round(100 * b.intersection(zona).area / b.area, 1) for b in barrios_m.geometry]

resultado

**Las preguntas, para responder por escrito:**

1. **¿Qué barrio queda mejor cubierto?** ¿Coincide con lo que mostraba la densidad por km² del
   bloque 8, o dice otra cosa? Si dice otra cosa, ¿por qué puede ser?
2. **Cambiá el radio** a 300 y a 1.000 metros. ¿En qué punto se invierte el orden entre los
   dos barrios, si es que se invierte? ¿Qué te dice eso sobre el peso de esa decisión?
3. **La cobertura que calculaste es de superficie.** Escribí en una frase qué habría que tener
   para calcularla sobre población, y por qué el número cambiaría.
4. **Una limitación de la fuente.** Mirá qué escuelas trae OSM en cada barrio: ¿distingue
   públicas de privadas? ¿Qué le hace eso a tu conclusión?

**Entrega:** media carilla más la tabla de resultados, antes del próximo encuentro.

---

## 12. 📋 El trabajo final

A partir de hoy tenés todo lo necesario para empezar el trabajo con el que se aprueba el
seminario: una pregunta territorial propia, datos de fuentes abiertas y variables construidas
por vos.

La consigna completa está en
[`TRABAJO_FINAL.md`](https://github.com/renzoepolo/sig-ciencias-sociales/blob/main/TRABAJO_FINAL.md)
y también en el aula virtual.

En una línea: **una notebook de Colab que corra de principio a fin, con una pregunta
territorial, al menos dos fuentes abiertas, dos variables territoriales construidas, dos mapas
terminados y una sección honesta de limitaciones.**

Lo único que conviene hacer esta semana es lo primero: **elegir la pregunta y la unidad de
análisis**, y chequear que existan los datos. Traelo a la Clase 7 y lo miramos.

---

## 13. Cierre

### El recorrido de hoy

| Pregunta | Operación | Lo que hay que decidir | La variable que salió |
|---|---|---|---|
| ¿Cómo junto la tabla con la geometría? | `merge` por clave | Que el código es texto, no número | `perc_65_mas` |
| ¿Cómo paso de direcciones a puntos? | Geocodificación | Si la tasa de error es tolerable, y para quién | — |
| ¿Qué hay adentro? | `sjoin` + `groupby` | El predicado, y por qué normalizar | `farmacias_por_km2` |
| ¿Qué hay alrededor? | `buffer` + `union_all` + `intersection` | El radio, y en qué CRS se mide | `cobertura_500m_perc` |

### La idea para llevarse

Una variable territorial **no se observa: se construye**, y cada una lleva adentro decisiones
que no quedan escritas en la columna. El radio de 500 metros, el predicado `within`, la
elección de dividir por superficie y no por población: nada de eso está en el nombre
`cobertura_500m_perc`, y todo eso cambia el número.

Es la misma lección de la Clase 5, corrida un paso hacia atrás. Allá el mapa escondía
decisiones; acá las esconde el dato mismo. Declararlas es lo que separa un indicador de un
número.

### Glosario de la clase

| Término | Definición |
|---|---|
| **Variable territorial** | Atributo de una unidad del espacio que se calcula a partir de la posición relativa de dos o más capas. |
| **Unión por clave** (`merge`) | Emparejamiento de dos tablas por una columna común. No usa la geometría. |
| **Unión espacial** (`sjoin`) | Emparejamiento de dos capas por su posición relativa. No necesita atributos en común. |
| **Predicado espacial** | La relación que la unión espacial busca: `within`, `intersects`, `contains`. |
| **Geocodificación** | Conversión de una dirección o un nombre en coordenadas. |
| **Área de influencia** (*buffer*) | Zona que rodea a una geometría hasta una distancia dada. |
| **Disolver** (`union_all`, `dissolve`) | Fundir varias geometrías en una, eliminando superposiciones. |
| **Superposición** (`overlay`, `intersection`) | Operación que calcula la parte del espacio común a dos geometrías. |
| **Cobertura** | Porcentaje de la superficie de una unidad que cae dentro de un área de influencia. |

### Tres preguntas para autoevaluarse

1. Tenés una capa de radios censales y una planilla del censo con el código de radio. El
   `merge` devuelve 0 coincidencias y ningún error. ¿Cuál es la primera hipótesis?
2. Calculaste el porcentaje de cobertura de un barrio y te dio 137 %. ¿Qué paso te faltó?
3. Un informe dice que el barrio A tiene el triple de equipamiento que el barrio B, y los dos
   datos salen de OpenStreetMap. ¿Qué preguntarías antes de creerlo?

### Qué viene en la Clase 7

Hoy medimos *adentro* y *alrededor*. Falta la tercera pregunta: **¿a qué distancia?**

La semana que viene calculamos la distancia al efector más cercano, y después la comparamos
con la distancia real caminando por las calles, que no es la misma. Vamos a ver cuánto
sobreestima el círculo de 500 metros que dibujamos hoy. Y vamos a resolver el problema que
quedó abierto en la interpretación del bloque 9: **cómo pasar una variable de una división
territorial a otra** —de radios censales a barrios— para poder calcular cobertura sobre
población y no sobre superficie.

---

## 14. Referencias

- de Smith, M. J., Goodchild, M. F. y Longley, P. A. (2018). *Geospatial Analysis: A
  Comprehensive Guide to Principles, Techniques and Software Tools* (6.ª ed.), caps. 4 y 7.
- Rey, S., Arribas-Bel, D. y Wolf, L. J. (2023). *Geographic Data Science with Python*,
  cap. 8 "Spatial Feature Engineering". CRC Press.
- INDEC (2023). *Censo Nacional de Población, Hogares y Viviendas 2022. Resultados
  definitivos*.
- Instituto Geográfico Nacional (2024). *Capas de Sistema de Información Geográfica —
  Salud*. Geoservicio WFS.
- Ministerio de Salud del Gobierno de la Ciudad de Buenos Aires. *Centros de Salud y Acción
  Comunitaria (CeSAC)*.
- Haklay, M. (2010). "How good is volunteered geographical information? A comparative study
  of OpenStreetMap and Ordnance Survey datasets". *Environment and Planning B*, 37(4),
  682–703.